In [1]:
"""
DSC2107: Data Mining & Wrangling
Group 4: Kilimall Kenya Scraper (kilimall.co.ke)
Requirements: >= 500 products across 3 categories, 7-day time series.
"""

from datetime import datetime
import json
import os
import random
import re
import sqlite3
import time
import pandas as pd
import requests

# =====================================================================
# 1. ETHICAL CONFIGURATION & SETTINGS
# =====================================================================
# Section 4(c): User-Agent identifying our course & email
HEADERS = {
    "User-Agent": "DSC2107-Course-Project (student@ucu.ac.ug)",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Origin": "https://www.kilimall.co.ke",
    "Referer": "https://www.kilimall.co.ke/"
}

# Kilimall's public backend search API
API_ENDPOINT = "https://api.kilimall.com/ke/v1/goods/search"

# Section 3(c): At least 3 categories
CATEGORIES = [
    {"name": "Phones & Accessories", "keyword": "phones", "target_count": 180},
    {"name": "Electronics & Audio", "keyword": "audio", "target_count": 180},
    {"name": "Computers & Laptops", "keyword": "laptops", "target_count": 180}
]

DB_NAME = "kilimall_data.db"

# =====================================================================
# 2. DATABASE INITIALIZATION (Part B: Relational SQLite Storage)
# =====================================================================
def init_database():
    """Creates the two normalized relational tables required by Section 3(c)."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    # Table 1: Dimension table (one row per unique product)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS products (
        product_id TEXT PRIMARY KEY,
        name TEXT NOT NULL,
        category TEXT NOT NULL,
        url TEXT
    )
    """)
    
    # Table 2: Fact table (longitudinal price/stock observations per day)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS observations (
        observation_id INTEGER PRIMARY KEY AUTOINCREMENT,
        product_id TEXT NOT NULL,
        price REAL NOT NULL,
        original_price REAL,
        discount_percent REAL,
        rating REAL,
        units_sold INTEGER,
        scraped_at TEXT NOT NULL,
        FOREIGN KEY (product_id) REFERENCES products(product_id)
    )
    """)
    
    conn.commit()
    conn.close()

# =====================================================================
# 3. ROBUST FETCH WITH RETRIES & BACK-OFF (Section 3d)
# =====================================================================
def fetch_api_page(keyword, page=1, retries=3):
    """
    Sends requests to Kilimall's API with retries and exponential back-off.
    """
    params = {
        "keyword": keyword,
        "page": page,
        "page_size": 40,
        "sort": 0
    }
    
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(API_ENDPOINT, headers=HEADERS, params=params, timeout=12)
            if response.status_code == 200:
                data = response.json()
                if data.get("code") == 200 or "data" in data:
                    return data
                else:
                    print(f"   [API Notice] Response returned message: {data.get('message', 'No data')}")
                    return None
            elif response.status_code == 429:
                print(f"   [Rate Limited] 429 Too Many Requests. Cooling down for {attempt * 10}s...")
                time.sleep(attempt * 10)
            else:
                print(f"   [Warning] Status {response.status_code} on attempt {attempt}.")
        except Exception as e:
            print(f"   [Error] Network error on attempt {attempt}: {e}")
        
        time.sleep(2 * attempt)
        
    return None

# =====================================================================
# 4. SCRAPING ENGINE
# =====================================================================
def scrape_kilimall():
    init_database()
    run_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    today_date = datetime.now().strftime("%Y%m%d")
    
    all_scraped_items = []
    print(f"=== Starting Kilimall Scraping Session at {run_timestamp} ===")

    for cat in CATEGORIES:
        category_name = cat["name"]
        keyword = cat["keyword"]
        target = cat["target_count"]
        
        print(f"\n📂 Scraping Category: '{category_name}' (Target: {target} items)")
        cat_collected = 0
        page = 1
        
        while cat_collected < target:
            print(f"   Fetching Page {page} for '{keyword}'...")
            result = fetch_api_page(keyword=keyword, page=page)
            
            if not result or not result.get("data"):
                print("   No more items returned. Moving to next category.")
                break
            
            # Extract goods list from response
            items = result.get("data", {}).get("goods_list", [])
            if not items:
                # Alternative structure fallback
                items = result.get("data", {}).get("list", [])
                
            if not items:
                print("   Items list is empty. Finished this category.")
                break

            for item in items:
                try:
                    product_id = str(item.get("goods_id") or item.get("id"))
                    name = item.get("goods_name") or item.get("title") or "Unknown Product"
                    
                    # 1. Price
                    price = float(item.get("goods_price") or item.get("price") or 0.0)
                    
                    # 2. Original Price & Discount %
                    market_price = float(item.get("market_price") or item.get("original_price") or price)
                    if market_price > price and market_price > 0:
                        discount = round(((market_price - price) / market_price) * 100, 2)
                    else:
                        discount = 0.0
                    
                    # 3. Rating
                    rating = float(item.get("score") or item.get("rate") or item.get("eval_score") or 0.0)
                    
                    # 4. Units Sold
                    units_sold = int(item.get("total_sales") or item.get("sales_num") or item.get("orderCount") or 0)
                    
                    product_url = f"https://www.kilimall.co.ke/new/goods/{product_id}"

                    record = {
                        "product_id": product_id,
                        "name": name.strip(),
                        "category": category_name,
                        "price": price,
                        "original_price": market_price,
                        "discount_percent": discount,
                        "rating": rating,
                        "units_sold": units_sold,
                        "product_url": product_url,
                        "scraped_at": run_timestamp
                    }
                    
                    all_scraped_items.append(record)
                    cat_collected += 1
                    
                    if cat_collected >= target:
                        break

                except Exception as parse_err:
                    continue

            print(f"   -> Progress: {cat_collected}/{target} items collected in '{category_name}'.")
            page += 1
            
            # Rule 4(b): Mandatory polite delay of 2 to 5 seconds
            sleep_time = random.uniform(2.0, 4.5)
            time.sleep(sleep_time)

    print(f"\n✅ Total Products Collected: {len(all_scraped_items)}")
    
    if len(all_scraped_items) == 0:
        print("No items collected. Check internet connection or API response format.")
        return

    # =====================================================================
    # 5. STORAGE: SAVE RAW EVIDENCE & LOAD TO RELATIONAL DB (Part B)
    # =====================================================================
    os.makedirs("data/raw", exist_ok=True)
    
    # 1. Save Raw JSON and CSV Evidence
    raw_csv_path = f"data/raw/kilimall_{today_date}_raw.csv"
    raw_json_path = f"data/raw/kilimall_{today_date}_raw.json"
    
    df = pd.DataFrame(all_scraped_items)
    df.to_csv(raw_csv_path, index=False)
    
    with open(raw_json_path, "w", encoding="utf-8") as f:
        json.dump(all_scraped_items, f, indent=2, ensure_ascii=False)
        
    print(f"📁 Raw evidence saved:")
    print(f"   - CSV:  {raw_csv_path}")
    print(f"   - JSON: {raw_json_path}")

    # 2. Ingest into Relational SQLite Database
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    for item in all_scraped_items:
        # Upsert into products table (ignore if already exists)
        cursor.execute("""
            INSERT OR IGNORE INTO products (product_id, name, category, url)
            VALUES (?, ?, ?, ?)
        """, (item["product_id"], item["name"], item["category"], item["product_url"]))
        
        # Insert time-series record into observations table
        cursor.execute("""
            INSERT INTO observations (product_id, price, original_price, discount_percent, rating, units_sold, scraped_at)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            item["product_id"],
            item["price"],
            item["original_price"],
            item["discount_percent"],
            item["rating"],
            item["units_sold"],
            item["scraped_at"]
        ))
        
    conn.commit()
    conn.close()
    print(f"💾 Cleaned data loaded into SQLite database: '{DB_NAME}' successfully!")
    
    # Quick sanity check print
    print("\nSample Preview (First 3 products):")
    print(df[["product_id", "category", "name", "price", "discount_percent", "units_sold"]].head(3))

if __name__ == "__main__":
    scrape_kilimall()

=== Starting Kilimall Scraping Session at 2026-10-13 13:18:07 ===

📂 Scraping Category: 'Phones & Accessories' (Target: 180 items)
   Fetching Page 1 for 'phones'...
   [Error] Network error on attempt 1: HTTPSConnectionPool(host='api.kilimall.com', port=443): Max retries exceeded with url: /ke/v1/goods/search?keyword=phones&page=1&page_size=40&sort=0 (Caused by NameResolutionError("HTTPSConnection(host='api.kilimall.com', port=443): Failed to resolve 'api.kilimall.com' ([Errno 11001] getaddrinfo failed)"))
   [Error] Network error on attempt 2: HTTPSConnectionPool(host='api.kilimall.com', port=443): Max retries exceeded with url: /ke/v1/goods/search?keyword=phones&page=1&page_size=40&sort=0 (Caused by NameResolutionError("HTTPSConnection(host='api.kilimall.com', port=443): Failed to resolve 'api.kilimall.com' ([Errno 11001] getaddrinfo failed)"))
   [Warning] Status 404 on attempt 3.
   No more items returned. Moving to next category.

📂 Scraping Category: 'Electronics & Audio' (Targe

In [2]:
import requests
from bs4 import BeautifulSoup

url = "https://www.kilimall.co.ke/search?q=phone&page=1"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

resp = requests.get(url, headers=headers)
print("Status Code:", resp.status_code)
print("Page length:", len(resp.text))

# Check if Samsung Galaxy A06 (from your screenshot) is in the HTML text!
if "Galaxy A06" in resp.text:
    print("🎉 SUCCESS: The products are directly in the HTML! We can scrape it using BeautifulSoup!")
else:
    print("⚡ The page uses dynamic JavaScript. We will use the network endpoint.")

Status Code: 200
Page length: 286773
🎉 SUCCESS: The products are directly in the HTML! We can scrape it using BeautifulSoup!


In [3]:
import re
import requests
from bs4 import BeautifulSoup

url = "https://www.kilimall.co.ke/search?q=phone&page=1"
headers = {
    "User-Agent": "DSC2107-Course-Project (student@ucu.ac.ug)"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

# Find all links pointing to goods/products
goods_links = soup.find_all("a", href=lambda h: h and "/new/goods/" in h)

print(f"Total product links discovered: {len(goods_links)}")

if goods_links:
    # Look at the first product card
    sample_card = goods_links[0]
    print("\n--- SAMPLE PRODUCT CARD FOUND ---")
    print("Product Link:", sample_card.get("href"))
    print("Raw Card Text:\n", sample_card.get_text(separator=" | ", strip=True))
else:
    # Alternative check: find elements containing 'KSh'
    print("\nSearching for price tags containing 'KSh'...")
    ksh_tags = soup.find_all(string=re.compile(r"KSh", re.I))
    print(f"Found {len(ksh_tags)} price mentions!")
    for tag in ksh_tags[:5]:
        print("-> Price text:", tag.strip(), "| Parent tag:", tag.parent.name, "| Classes:", tag.parent.get("class"))

Total product links discovered: 0

Searching for price tags containing 'KSh'...
Found 37 price mentions!
-> Price text: KSh 13,999 | Parent tag: div | Classes: ['product-price']
-> Price text: KSh 16,499 | Parent tag: div | Classes: ['product-price']
-> Price text: KSh 20,899 | Parent tag: div | Classes: ['product-price']
-> Price text: KSh 19,599 | Parent tag: div | Classes: ['product-price']
-> Price text: KSh 10,999 | Parent tag: div | Classes: ['product-price']


In [4]:
# Grab the very first price tag
first_price_tag = ksh_tags[0]

# Find the entire card containing this product
# We walk up to the main card container
card = first_price_tag.find_parent("div", class_=lambda c: c and any("product" in x or "goods" in x or "item" in x for x in c))
if not card:
    card = first_price_tag.parent.parent  # Walk two levels up

print("=== ENTIRE PRODUCT CARD TEXT ===")
print(card.get_text(separator=" | ", strip=True))

print("\n=== ALL HTML TAGS & CLASSES INSIDE THIS CARD ===")
for elem in card.find_all(True):
    classes = elem.get("class")
    if classes:
        print(f"Tag: <{elem.name}> | Classes: {classes} | Text snippet: {elem.text[:30].strip()}")

=== ENTIRE PRODUCT CARD TEXT ===
Brand New Samsung Galaxy A06 64GB+4GB 50MP Camera 6.7" 90Hz MTK Helio G85 (12nm) 5000mAh 25W Type-C Charge Android 14 Smart Phones phone Samsung A06 | KSh 13,999 | (3254)

=== ALL HTML TAGS & CLASSES INSIDE THIS CARD ===
Tag: <p> | Classes: ['product-title'] | Text snippet: Brand New Samsung Galaxy A06
Tag: <img> | Classes: ['choice-tag'] | Text snippet: 
Tag: <div> | Classes: ['product-price'] | Text snippet: KSh 13,999
Tag: <div> | Classes: ['rate'] | Text snippet: (3254)
Tag: <div> | Classes: ['van-rate'] | Text snippet: 
Tag: <div> | Classes: ['van-rate__item'] | Text snippet: 
Tag: <i> | Classes: ['van-badge__wrapper', 'van-icon', 'van-icon-star', 'van-rate__icon', 'van-rate__icon--full'] | Text snippet: 
Tag: <div> | Classes: ['van-rate__item'] | Text snippet: 
Tag: <i> | Classes: ['van-badge__wrapper', 'van-icon', 'van-icon-star', 'van-rate__icon', 'van-rate__icon--full'] | Text snippet: 
Tag: <div> | Classes: ['van-rate__item'] | Text snippet: 


In [5]:
"""
DSC2107: Data Mining & Wrangling
Group 4: Kilimall Kenya Scraper (kilimall.co.ke)
Method: HTML Parsing with requests + BeautifulSoup (Server-Side Rendered)
Target: >= 500 products across 3 categories per run, with SQLite storage.
"""

from datetime import datetime
import json
import os
import random
import re
import sqlite3
import time
from urllib.parse import urljoin
import pandas as pd
import requests
from bs4 import BeautifulSoup

# =====================================================================
# 1. ETHICAL CONFIGURATION (Section 4 Compliance)
# =====================================================================
HEADERS = {
    "User-Agent": "DSC2107-Course-Project (student@ucu.ac.ug)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9"
}

BASE_SEARCH_URL = "https://www.kilimall.co.ke/search"

CATEGORIES = [
    {"name": "Phones & Accessories", "query": "phone", "target_items": 180},
    {"name": "Audio & Electronics", "query": "audio", "target_items": 180},
    {"name": "Computers & Laptops", "query": "laptops", "target_items": 180}
]

DB_NAME = "kilimall_data.db"

# =====================================================================
# 2. DATABASE SETUP (Part B: Relational SQLite Storage)
# =====================================================================
def init_database():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    # Table 1: Dimension table for static product data
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS products (
        product_id TEXT PRIMARY KEY,
        name TEXT NOT NULL,
        category TEXT NOT NULL,
        product_url TEXT
    )
    """)
    
    # Table 2: Longitudinal Fact table for time-series observations
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS observations (
        observation_id INTEGER PRIMARY KEY AUTOINCREMENT,
        product_id TEXT NOT NULL,
        price REAL NOT NULL,
        original_price REAL,
        discount_percent REAL,
        rating REAL,
        units_sold INTEGER,
        scraped_at TEXT NOT NULL,
        FOREIGN KEY (product_id) REFERENCES products(product_id)
    )
    """)
    conn.commit()
    conn.close()

# =====================================================================
# 3. CARD EXTRACTION HELPER
# =====================================================================
def parse_product_card(card, category_name, timestamp):
    """
    Extracts all 5 required fields from a single Kilimall product card.
    """
    try:
        # 1. Product Title
        title_tag = card.find("p", class_="product-title")
        if not title_tag:
            return None
        title = title_tag.text.strip()

        # 2. Price (e.g. 'KSh 13,999' -> 13999.0)
        price_tag = card.find("div", class_="product-price")
        if not price_tag:
            return None
        price_clean = re.sub(r"[^\d.]", "", price_tag.text)
        if not price_clean:
            return None
        price = float(price_clean)

        # 3. Rating (Count full stars inside the van-rate component)
        # Kilimall uses class 'van-rate__icon--full' for active stars
        full_stars = card.find_all(class_=re.compile(r"van-rate__icon--full"))
        half_stars = card.find_all(class_=re.compile(r"van-rate__icon--half"))
        rating = float(len(full_stars) + (0.5 * len(half_stars)))

        # 4. Units Sold / Reviews (e.g. '(3254)' -> 3254)
        reviews_tag = card.find("span", class_="reviews")
        if reviews_tag:
            review_clean = re.sub(r"[^\d]", "", reviews_tag.text)
            units_sold = int(review_clean) if review_clean else 0
        else:
            units_sold = 0

        # 5. Old Price & Discount %
        # Look for strikethrough or secondary price tags inside the card
        old_price_tag = card.find(class_=re.compile(r"origin|old|market", re.I))
        if old_price_tag:
            old_price_clean = re.sub(r"[^\d.]", "", old_price_tag.text)
            old_price = float(old_price_clean) if old_price_clean else price
        else:
            old_price = price

        if old_price > price:
            discount_pct = round(((old_price - price) / old_price) * 100, 2)
        else:
            discount_pct = 0.0

        # 6. Product Link & ID
        link_tag = card.find("a") or card.find_parent("a")
        if link_tag and link_tag.get("href"):
            raw_url = link_tag.get("href")
            product_url = urljoin("https://www.kilimall.co.ke", raw_url)
            # Try to grab numeric ID from URL (e.g. /goods/12345)
            id_match = re.search(r"(\d{5,})", raw_url)
            product_id = id_match.group(1) if id_match else f"prod_{abs(hash(title)) % 10000000}"
        else:
            product_id = f"prod_{abs(hash(title)) % 10000000}"
            product_url = f"https://www.kilimall.co.ke/search?q={title[:20]}"

        return {
            "product_id": str(product_id),
            "name": title,
            "category": category_name,
            "price": price,
            "original_price": old_price,
            "discount_percent": discount_pct,
            "rating": rating,
            "units_sold": units_sold,
            "product_url": product_url,
            "scraped_at": timestamp
        }
    except Exception as e:
        return None

# =====================================================================
# 4. MAIN SCRAPING CONTROLLER
# =====================================================================
def run_kilimall_pipeline():
    init_database()
    run_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    today_str = datetime.now().strftime("%Y%m%d")
    
    all_products = []
    print(f"=== Starting Kilimall Live Scraping Run at {run_timestamp} ===")

    for cat in CATEGORIES:
        category_name = cat["name"]
        query = cat["query"]
        target = cat["target_items"]
        
        print(f"\n📂 Category: '{category_name}' (Target: {target} items)")
        collected_in_cat = 0
        page = 1
        
        while collected_in_cat < target:
            page_url = f"{BASE_SEARCH_URL}?q={query}&page={page}"
            print(f"   Fetching Page {page}: {page_url}")
            
            try:
                resp = requests.get(page_url, headers=HEADERS, timeout=15)
                if resp.status_code != 200:
                    print(f"   [Notice] Non-200 status code: {resp.status_code}. Stopping category.")
                    break
            except Exception as net_err:
                print(f"   [Error] Connection error: {net_err}")
                break

            soup = BeautifulSoup(resp.text, "html.parser")
            
            # Find all product containers
            # Each product card contains a <p class="product-title">
            title_tags = soup.find_all("p", class_="product-title")
            
            if not title_tags:
                print(f"   No more products found on page {page}. Ending category.")
                break

            page_new_count = 0
            for t_tag in title_tags:
                # Walk up to the parent product box container
                card_container = t_tag.find_parent("div", class_=lambda c: c and any(x in str(c) for x in ["product", "item", "goods", "box"]))
                if not card_container:
                    card_container = t_tag.parent.parent
                
                item_data = parse_product_card(card_container, category_name, run_timestamp)
                if item_data:
                    all_products.append(item_data)
                    collected_in_cat += 1
                    page_new_count += 1
                    if collected_in_cat >= target:
                        break

            print(f"   -> Added {page_new_count} items from page {page}. Total in category: {collected_in_cat}/{target}")
            page += 1

            # Rule 4(b): Polite randomized delay between requests (2 to 4 seconds)
            time.sleep(random.uniform(2.0, 4.0))

    print(f"\n==========================================")
    print(f"🎉 Scraping Finished! Total Products: {len(all_products)}")
    print(f"==========================================")

    if not all_products:
        print("No products collected. Check selectors or connectivity.")
        return

    # =====================================================================
    # 5. STORAGE & DATABASE LOADING (Part B Compliance)
    # =====================================================================
    os.makedirs("data/raw", exist_ok=True)
    os.makedirs("data/processed", exist_ok=True)
    
    # 1. Save Unmodified Raw CSV & JSON Files (Deliverable 4)
    raw_csv_path = f"data/raw/kilimall_raw_{today_str}.csv"
    raw_json_path = f"data/raw/kilimall_raw_{today_str}.json"
    
    df = pd.DataFrame(all_products)
    df.to_csv(raw_csv_path, index=False)
    
    with open(raw_json_path, "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=2, ensure_ascii=False)

    print(f"📁 Evidence saved:")
    print(f"   - CSV:  {raw_csv_path}")
    print(f"   - JSON: {raw_json_path}")

    # 2. Ingest into Relational SQLite Database
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    for item in all_products:
        # Upsert into products (dimension) table
        cursor.execute("""
            INSERT OR IGNORE INTO products (product_id, name, category, product_url)
            VALUES (?, ?, ?, ?)
        """, (item["product_id"], item["name"], item["category"], item["product_url"]))
        
        # Insert observation into time-series table
        cursor.execute("""
            INSERT INTO observations (product_id, price, original_price, discount_percent, rating, units_sold, scraped_at)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            item["product_id"],
            item["price"],
            item["original_price"],
            item["discount_percent"],
            item["rating"],
            item["units_sold"],
            item["scraped_at"]
        ))
        
    conn.commit()
    conn.close()
    print(f"💾 Ingested into SQLite Database: '{DB_NAME}' successfully!")
    
    # Preview top 5 rows
    print("\nPreview of Scraped Data:")
    print(df[["product_id", "category", "name", "price", "rating", "units_sold"]].head())

if __name__ == "__main__":
    run_kilimall_pipeline()

=== Starting Kilimall Live Scraping Run at 2026-10-13 13:51:44 ===

📂 Category: 'Phones & Accessories' (Target: 180 items)
   Fetching Page 1: https://www.kilimall.co.ke/search?q=phone&page=1
   -> Added 36 items from page 1. Total in category: 36/180
   Fetching Page 2: https://www.kilimall.co.ke/search?q=phone&page=2
   -> Added 36 items from page 2. Total in category: 72/180
   Fetching Page 3: https://www.kilimall.co.ke/search?q=phone&page=3
   -> Added 36 items from page 3. Total in category: 108/180
   Fetching Page 4: https://www.kilimall.co.ke/search?q=phone&page=4
   -> Added 36 items from page 4. Total in category: 144/180
   Fetching Page 5: https://www.kilimall.co.ke/search?q=phone&page=5
   -> Added 36 items from page 5. Total in category: 180/180

📂 Category: 'Audio & Electronics' (Target: 180 items)
   Fetching Page 1: https://www.kilimall.co.ke/search?q=audio&page=1
   -> Added 36 items from page 1. Total in category: 36/180
   Fetching Page 2: https://www.kilimall.co.k

OperationalError: table products has no column named product_url